# Análise de Acessibilidade

**AUTORIA:** [REDE MOB](https://www.redemob.com.br/)

Análise do padrão de viagens de um município com base em dados de pesquisa de origem e destino. As linhas de código abaixo têm como objeto o município de Belo Horizonte, mas serve para qualquer outro, desde que os dados de entrada estejam no layout padrão da Plataforma.

**PANORAMA:**
- #TODO: Listar as análises aqui contidas

**MAIS INFORMAÇÕES:**
- [Layout da Plataforma]
- [Sumário dos Dados Disponíveis]
- *Lorem ipsum: Conteúdo do MOB de interesse, técnico ou de divulgação*

**LINKS DE INTERESSE:**
- links para materiais técnicos e acadêmicos gerais de referência a respeito do conteúdo abordado



# Introdução

Este script foi concebido em caráter de tutorial, tomando como exemplo o município de Belo Horizonte/MG, de forma que as considerações e discussões aqui contidas foram tecidas no contexto dessa municipalidade. Com efeito, procurou-se, na medida do possível, deixar o texto abrangente a ponto de orientar as análises de outros municípios. Ou seja, este script acaba constituindo um apoio para o diagnóstico de outras localidade, na medida em que houve um esforço de trazer técnicas e elementos norteadores para contribuir para o diagnóstico de outros locais. Com efeito, ao alterar os parâmetros de entrada, conforme demonstrado logo abaixo, podem ser gerados mapas e gráficos de territórios distintos. Nesse caso, as considerações originalmente tecidas para Belo Horizonte podem não se aplicar completamente, mas, elas ainda devem fornecer insumos para a interpretação de resultados de outros locais.

# Instruções

Este script precisa de dois grupos de dados:
1. Dados de origem e destino #TODO: explicar como obter
2. Malhas territoriais de zonas de tráfego

Essses dados devem ser atribuídos às variáveis abaixo. Em seguida, deve-se rodar todo o script e os resultados estarão ao final.

# Backend

In [1]:
import datetime as dt
from joblib import Parallel, delayed
import os
import pathlib
from pathlib import Path
import re
import subprocess
import warnings

import geobr
import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import partridge as ptg
from pyrosm import get_data
import r5py
import seaborn as sns
from shapely.geometry import LineString
from tqdm.auto import tqdm

In [2]:
%matplotlib inline
%config InlineBackend.figure_format='retina'

pd.options.display.float_format = '{:,.2f}'.format

tqdm_bar_format = (
    "{desc:<12} {percentage:3.0f}%|{bar}| "
    "{n_fmt}/{total_fmt} • {rate_fmt} • {elapsed}<{remaining} {postfix}"
)

In [3]:
out_folder = os.environ.get('OUT_FOLDER')
out_folder = pathlib.Path(out_folder)

db_folder = os.environ.get('DB_FOLDER')
db_folder = pathlib.Path(db_folder)

# Consertando o GTFS

Os GTFS fornecidos pela prefeituea de BH, após solicitação, utilizam apenas a tabela calendar_dates.txt, em vez do calendar. Isso pode gerar alguns problemas nos cálculos com o R5Py. A primeira ideia foi criar um calendário completo a partir do calendar_dates. Mas, em alguns casos, nem todas os serviços presentes em trips.txt estão em calendar_dates.txt, ao mesmo tempo em que as datas presentes são relativamente erráticas, muitas vezes, inclusive, muitas vezes se concentrando em feriados e fins de semana. Isso aponta para uma certa limitação do dado e que é de difícil correção a essa altura, sobretudo porque trata-se de dados históricos, coletados anos atrás, bem antes da pandemia.

Nesse sentido, uma saída razoável é, para cada ano, selecionar a data com mais serviços em operação e tomar esse como o dia representativo. Mesmo que, dado o exposto acima, a acessibilidade resultante esteja em alguma medida subrepresentada, não há razão para não supor que a estrutura geral do acesso na cidade não estará constante. Isto é, é razoável partir da premissa de que os gradientes de acessibilidade ao longo do território são mantidos, o que não deve impactar o modelo de forma muito relevante, sobretudo porque, para que esse dado de acessibilidade entre no modelo, ele será normalizado de zero a um.

Adicionalmente, há o problema de que não há os shapes para a maoria das linhas e os tempos de chegada e partida apenas existem nos terminais. Portanto, é necessário fazer uma interpolação manual para estimar os tempos intermediáris, a partir do tempo entre terminais e a distância em linha reta entre pontos consecutivos. Essa acaba sendo uma limitação quase incortonável da pesquisa, mas, como a mediana da distância entre pontos é 500m, entende-se que isso não é algo tão grave para o contexto do modelo, que simula toda a cidade de Belo Horizonte.

In [4]:
DEFAULT_CRS = 31983
DEFAULT_STOP_SPACING_SEC = 30
MAX_ALLOWED_SECONDS = 86399 + 3600 * 4


def format_time_column(seconds: np.ndarray) -> pd.Series:
    seconds = np.clip(seconds.astype(int), 0, MAX_ALLOWED_SECONDS)
    h, m = divmod(seconds, 3600)
    m, s = divmod(m, 60)
    return pd.Series([
        f"{int(hh):02}:{int(mm):02}:{int(ss):02}"
        for hh, mm, ss in zip(h, m, s)
    ]).to_numpy()


def interpolate_times(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values("stop_sequence").copy()
    arrival_time = group["arrival_time"].fillna("")
    arrival_time = arrival_time.where(
        arrival_time.str.match(r"^\d{2}:\d{2}:\d{2}$"), ""
    )
    time_deltas = pd.to_timedelta(arrival_time, errors="coerce")
    time_secs = time_deltas.dt.total_seconds()
    known = time_secs.notna()

    if known.sum() < 2:
        group["arrival_sec"] = (
            np.arange(len(group)) * DEFAULT_STOP_SPACING_SEC
        )
    else:
        start_sec, end_sec = time_secs[known].iloc[[0, -1]]
        if end_sec < start_sec:
            end_sec += 86400

        total_dist = group["shape_dist_traveled"].max()
        if total_dist:
            dist_ratio = group["shape_dist_traveled"] / total_dist
            duration = end_sec - start_sec
        else:
            dist_ratio = np.arange(len(group))
            duration = DEFAULT_STOP_SPACING_SEC

        group["arrival_sec"] = start_sec + dist_ratio * duration

    group["arrival_time"] = format_time_column(group["arrival_sec"])
    group["departure_time"] = group["arrival_time"]
    return group


def build_stop_times(raw_feed, feed) -> pd.DataFrame:
    stop_times = raw_feed.stop_times.copy()
    stop_times["stop_sequence"] = stop_times["stop_sequence"].astype(int)

    stops = feed.stops.to_crs(DEFAULT_CRS)
    gdf = (
        stops
        .merge(stop_times, on="stop_id")
        .sort_values(["trip_id", "stop_sequence"])
    )
    gdf["prev_geom"] = gdf.groupby("trip_id")["geometry"].shift()
    gdf["segment_distance"] = gdf["geometry"].distance(
        gdf["prev_geom"]
    ).fillna(0)

    gdf["shape_dist_traveled"] = gdf.groupby("trip_id")[
        "segment_distance"
    ].cumsum()

    interpolated = Parallel(n_jobs=-1)(
        delayed(interpolate_times)(group)
        for _, group in gdf.groupby("trip_id")
    )

    return pd.concat(interpolated, ignore_index=True).reindex(
        columns=list(stop_times.columns) + ["shape_dist_traveled"]
    )


def fix_gtfs(gtfs_zip_path: Path, output_dir: Path) -> None:
    raw_feed = ptg.load_raw_feed(gtfs_zip_path)
    geo_feed = ptg.load_geo_feed(
        gtfs_zip_path,
    )
    stop_times = build_stop_times(raw_feed, geo_feed)
    raw_feed.set("stop_times.txt", stop_times)

    output_dir.mkdir(parents=True, exist_ok=True)
    ptg.writers.write_feed_dangerously(
        raw_feed, str(output_dir / gtfs_zip_path.name)
    )


In [5]:
def date_ordered_files(gtfs_root):
    return sorted(
        gtfs_root.glob("*.zip"),
        key=lambda f: dt.datetime.strptime(f.stem.split("_")[1], "%d-%m-%Y")
        )

In [6]:
gtfs_root = db_folder / 'beaga/gtfs_estatico_coletanea'
output_dir = out_folder / 'B' / 'corrected_gtfs'

for gtfs_path in date_ordered_files(gtfs_root):
    fix_gtfs(gtfs_path, output_dir)
    print(
        f"{gtfs_path.stem}: Done!"
        )

c:\Users\brand\anaconda3\envs\roda\Lib\site-packages\numpy\lib\function_base.py:2455: RuntimeWarning: invalid value encountered in parse_time (vectorized)
  outputs = ufunc(*inputs)
c:\Users\brand\anaconda3\envs\roda\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


GTFSFiles_08-03-2011: Done!


c:\Users\brand\anaconda3\envs\roda\Lib\site-packages\numpy\lib\function_base.py:2455: RuntimeWarning: invalid value encountered in parse_time (vectorized)
  outputs = ufunc(*inputs)
c:\Users\brand\anaconda3\envs\roda\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


GTFSFiles_13-09-2011: Done!


c:\Users\brand\anaconda3\envs\roda\Lib\site-packages\numpy\lib\function_base.py:2455: RuntimeWarning: invalid value encountered in parse_time (vectorized)
  outputs = ufunc(*inputs)
c:\Users\brand\anaconda3\envs\roda\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


...

GTFSFiles_05-03-2020: Done!


c:\Users\brand\anaconda3\envs\roda\Lib\site-packages\numpy\lib\function_base.py:2455: RuntimeWarning: invalid value encountered in parse_time (vectorized)
  outputs = ufunc(*inputs)
c:\Users\brand\anaconda3\envs\roda\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


GTFSFiles_01-09-2020: Done!


c:\Users\brand\anaconda3\envs\roda\Lib\site-packages\numpy\lib\function_base.py:2455: RuntimeWarning: invalid value encountered in parse_time (vectorized)
  outputs = ufunc(*inputs)
c:\Users\brand\anaconda3\envs\roda\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


GTFSFiles_02-03-2021: Done!


In [7]:
gtfs_root = out_folder / 'B' / 'corrected_gtfs'

## General Tests